# Sentinel-2 Cloud-Free Composite Export — Paoay, Atok, Benguet

Builds **half-month (H1/H2) cloud-masked Sentinel-2 median composites** for the study area using Google Earth Engine (GEE), and exports each one to Google Drive as a GeoTIFF. Also includes a coverage check that reports how much of the area survives cloud masking in each period.

**Data sources**
- `COPERNICUS/S2_SR_HARMONIZED` — Sentinel-2 surface reflectance
- `GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED` — Cloud Score+ (per-pixel clear-sky score)

**Contents**
1. Setup — imports and Earth Engine authentication
2. Configuration — area of interest, years, thresholds, export settings
3. Helper functions — cloud masking
4. Map preview — visual sanity check
5. Export composites — main loop → Google Drive
6. Export summary — what was exported / skipped
7. Cloud coverage check — valid-pixel % per period
8. Spectral indices — NDVI, NDRE, NDMI from the exported composites → Google Drive
9. Feature stack — one table per year: one row per pixel, one column per band/index per period


> **Before running:** see `README.md` for environment setup. Change `GEE_PROJECT` in the configuration cell to your own Earth Engine project.

## 1. Setup

Imports and Earth Engine login. `ee.Authenticate()` opens a browser prompt — it is only strictly needed the first time (or when your token expires), after which `ee.Initialize()` alone is enough.

In [ ]:
import calendar

import ee
import geemap
import pandas as pd
from tqdm import tqdm

from pathlib import Path
import numpy as np
import rasterio

import matplotlib.pyplot as plt

import json
from rasterio.features import geometry_mask
from rasterio.warp import transform_geom


In [ ]:
GEE_PROJECT = "thesis-18"  # <-- replace with your own Earth Engine project ID

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
print(ee.String("Earth Engine is working!").getInfo())

## 2. Configuration

All tunable settings live here.

| Setting | Meaning |
|---|---|
| `paoay_aoi` | Study area rectangle `[west, south, east, north]` |
| `years` | Years to process |
| `CS_THRESHOLD` | Cloud Score+ cutoff (0–1). Higher = stricter → fewer but cleaner pixels |
| `BANDS` | Sentinel-2 bands kept in the export |
| `SCALE` | Export resolution in metres |
| `DRIVE_FOLDER` | Google Drive folder that receives the exports |

> **TODO:** replace the rectangle with the actual Paoay shapefile / boundary geometry.

### 2.1 Area of interest

In [ ]:
# Area of Interest — Paoay, Atok, Benguet
# TODO: replace with the actual Paoay boundary (shapefile) instead of a bounding rectangle
paoay_aoi = geemap.shp_to_ee("Paoay_SHP_FILE/Boundary_Paoay_ATOK.shp") 

### 2.2 Processing parameters

In [ ]:
years = [2019, 2020, 2021, 2022, 2023, 2024]

# Cloud Score+ collection (matched to source images by system:index)
cs_plus = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
CS_THRESHOLD = 0.6  # 0-1, higher = stricter (fewer, cleaner pixels)

# Export settings
BANDS = ["B3", "B4", "B5", "B8", "B11"]
SCALE = 10  # metres
DRIVE_FOLDER = "THS-SENTINEL2"
FILE_PREFIX = "Sentinel2_CSPlus"  # prefix for exported file names

SCENE_CLOUD_MAX = 80   # % (CLOUDY_PIXEL_PERCENTAGE)

# One output grid for EVERY export and pixel count
TARGET_CRS = "EPSG:32651"                   # WGS 84 / UTM zone 51N
TARGET_TRANSFORM = [10, 0, 0, 0, -10, 0]    # 10 m pixels, snapped to the native Sentinel-2 10 m grid
GRID = dict(crs=TARGET_CRS, crsTransform=TARGET_TRANSFORM)

### 2.3 Time periods

Each month is split into two windows: **H1** (1st–15th) and **H2** (16th–end of month), giving 24 periods per year.

In [ ]:
periods_list = []
for year in years:
    for month in range(1, 13):
        next_first = f"{year + 1}-01-01" if month == 12 else f"{year}-{month + 1:02d}-01"
        periods_list.append((year, month, f"{year}-{month:02d}-01", f"{year}-{month:02d}-16", "H1"))
        periods_list.append((year, month, f"{year}-{month:02d}-16", next_first, "H2"))
        
print(f"{len(periods_list)} periods to process")
print("example:", periods_list[0], periods_list[1], periods_list[23])

## 3. Helper functions

`mask_clouds_cs_plus` masks every pixel whose Cloud Score+ (`cs_cdf` band) is below `CS_THRESHOLD`. It is mapped over each image collection after linking with the Cloud Score+ collection.

In [ ]:
def mask_clouds_cs_plus(image):
    """Attach Cloud Score+ and mask pixels below the clear-sky threshold."""
    cs = image.select("cs_cdf")
    return image.updateMask(cs.gte(CS_THRESHOLD))

In [ ]:
def scene_collection(start_date, end_date):
    """The ONLY place scenes are selected. Used by Sections 5, 7, 8 and 9."""
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(paoay_aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", SCENE_CLOUD_MAX))
    )

## 4. Map preview

Creates the interactive map used as a visual sanity check. Composite layers are added to it during the export loop in Section 5 (hidden by default — toggle them from the layer control). The map is displayed at the end of Section 6.

In [ ]:
Map = geemap.Map()
Map.add_basemap("SATELLITE")
Map.add_layer(paoay_aoi, {"color": "red"}, "Study Area")
Map.centerObject(paoay_aoi, 13)

vis_params = {
    "bands": ["B4", "B3", "B5"],
    "min": 0,
    "max": 3000,
}

## 5. Export composites

For every period this cell does the following:

1. **Filter** Sentinel-2 SR scenes by area, date, and scene-level cloud % (< SCENE_CLOUD_MAX, set in 2.2)
2. **Mask** clouds per pixel with Cloud Score+
3. **Composite** using the per-pixel median of clear pixels, clipped to the AOI
4. **Skip** the period if there is no imagery, or if every pixel was masked
5. **Export** the composite to Google Drive (`DRIVE_FOLDER`) as `Sentinel2_CSPlus_<year>_<month>_<H1|H2>`

> ⏱ This loop calls `.getInfo()` several times per period, so it takes a while. Exports run as background GEE tasks — track them at https://code.earthengine.google.com/tasks.

In [ ]:
summary = []

for year, month, start_date, end_date, half in periods_list:
    label = f"{year}_{month:02d}_{half}"
    print(f"\n{'='*60}\nPROCESSING {label}\n{'='*60}")

    # Raw collection for this window
    collection = scene_collection(start_date, end_date)

    raw_count = collection.size().getInfo()
    print(f"  Raw scenes found: {raw_count}")

    if raw_count == 0:
        print("  → Skipped: no imagery at all this period")
        summary.append({"label": label, "raw_count": 0, "status": "no_imagery"})
        continue

    # Attach Cloud Score+ and mask
    masked_collection = collection.linkCollection(cs_plus, ["cs_cdf"]).map(mask_clouds_cs_plus)

    # Composite: per-pixel median of clear pixels only
    composite = masked_collection.median().clip(paoay_aoi)
    raw_bands = composite.select(BANDS)

    # Check how much of the AOI actually has valid (non-masked) data
    valid_pixel_pct = raw_bands.select("B4").reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=paoay_aoi,
        **GRID,
        maxPixels=1e13,
    ).get("B4")

    valid_count = ee.Number(valid_pixel_pct).getInfo()
    print(f"  Valid (unmasked) pixel count in B4: {valid_count}")

    if valid_count == 0:
        print("  → Skipped: fully masked, no clear pixels survived")
        summary.append({"label": label, "raw_count": raw_count, "status": "fully_masked"})
        continue

    # Add to map for visual check
    Map.addLayer(raw_bands, vis_params, f"S2 {label}", False)  # False = off by default, toggle manually

    # Export
    task = ee.batch.Export.image.toDrive(
        image=raw_bands,
        description=f"{FILE_PREFIX}_{label}",
        folder=DRIVE_FOLDER,
        fileNamePrefix=f"{FILE_PREFIX}_{label}",
        region=paoay_aoi.geometry(),
        **GRID,
        maxPixels=1e13,
    )
    task.start()
    print(f"  → Export started: {FILE_PREFIX}_{label}")

    summary.append({"label": label, "raw_count": raw_count, "status": "exported"})

## 6. Export summary

Counts how many periods were exported vs. skipped, lists the problem periods, and shows the map.

In [ ]:
print("=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)

exported = [s for s in summary if s["status"] == "exported"]
skipped_empty = [s for s in summary if s["status"] == "no_imagery"]
skipped_masked = [s for s in summary if s["status"] == "fully_masked"]

print(f"Exported: {len(exported)}")
print(f"Skipped (no imagery): {len(skipped_empty)}")
print(f"Skipped (fully masked): {len(skipped_masked)}")

if skipped_empty or skipped_masked:
    print("\nProblem periods:")
    for s in skipped_empty + skipped_masked:
        print(f"  - {s['label']} ({s['status']})")

In [ ]:
display(Map)

## 7. Cloud coverage check

Independent of the export: for each period, computes the **percentage of AOI pixels that remain valid after Cloud Score+ masking**, and saves the result to `cloudscore_coverage_check.csv`.

| Status | Meaning |
|---|---|
| `good` | ≥ 80 % valid pixels |
| `partial` | 0–80 % valid pixels |
| `fully_masked` | no valid pixels |
| `no_imagery` | no scenes found for the period |

Uses the same scenes as the export (scene_collection, CLOUDY_PIXEL_PERCENTAGE < SCENE_CLOUD_MAX) and the same EPSG:32651 10 m grid, so the coverage describes exactly the observations in the exported files.

In [ ]:
results = []

for year, month, start_date, end_date, half in tqdm(periods_list, desc="Checking coverage"):
    label = f"{year}_{month:02d}_{half}"

    # Loose scene-level filter — just to skip obviously hopeless scenes
    collection = scene_collection(start_date, end_date)
    raw_count = collection.size().getInfo()

    if raw_count == 0:
        results.append({"label": label, "raw_count": 0, "valid_pixel_pct": 0, "status": "no_imagery"})
        continue

    # Apply real per-pixel cloud masking
    masked_collection = collection.linkCollection(cs_plus, ["cs_cdf"]).map(mask_clouds_cs_plus)
    composite = masked_collection.median().clip(paoay_aoi).select("B4")

    # Count valid (non-masked) pixels vs total pixels in AOI
    valid_count = ee.Number(composite.reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=paoay_aoi,
        **GRID,
        maxPixels=1e13,
    ).get("B4")).getInfo()

    total_count = ee.Number(ee.Image(1).clip(paoay_aoi).reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=paoay_aoi,
        **GRID,
        maxPixels=1e13,
    ).get("constant")).getInfo()

    valid_pct = round((valid_count / total_count) * 100, 1) if total_count > 0 else 0

    status = "good" if valid_pct >= 80 else ("partial" if valid_pct > 0 else "fully_masked")

    results.append({
        "label": label,
        "raw_count": raw_count,
        "valid_pixel_pct": valid_pct,
        "status": status,
    })

df = pd.DataFrame(results)
df.to_csv("cloudscore_coverage_check.csv", index=False)

print("\nSaved to cloudscore_coverage_check.csv\n")

### 7.1 Results

In [ ]:
print("Status breakdown:")
print(df["status"].value_counts())

print("\nWorst periods (lowest valid pixel %):")
print(df.sort_values("valid_pixel_pct").head(15)[["label", "raw_count", "valid_pixel_pct", "status"]])

## 8. Spectral indices (NDVI, NDRE, NDMI ) 

This calculates three indices from the median composite bands.

### 8.1 Define the indices and output settings

- INDEX_DEFS matches each index name to the two bands used in its formula.
- FEATURES combines the five names in BANDS with the three index names. This gives eight features for each half-month.
- DRIVE_FOLDER_INDICES and INDEX_PREFIX set the folder and filename prefix for the separate index exports.
- DRIVE_FOLDER_FEATURES and STACK_PREFIX set the folder and filename prefix for the yearly stacks in Section 9.
- NODATA = -9999.0 is the export marker for missing data. Zero remains a possible real index value.

In [ ]:
INDEX_DEFS = {
    "NDVI": ("B8", "B4"),   # NIR, Red
    "NDRE": ("B8", "B5"),   # NIR, Red Edge 1
    "NDMI": ("B8", "B11"),  # NIR, SWIR1
}
FEATURES = BANDS + list(INDEX_DEFS)          # 8 features per period

DRIVE_FOLDER_INDICES  = "THS-SENTINEL2-INDICES"
INDEX_PREFIX          = "S2_Indices"
DRIVE_FOLDER_FEATURES = "THS-SENTINEL2-FEATURES"
STACK_PREFIX          = "S2_FeatureStack"
                
NODATA = -9999.0                             # indices can be 0, so 0 can't mean "masked"

Builds one half-month band composite. start_date and end_date specify the period to process. The function returns a five-band Earth Engine image.

In [ ]:
def make_composite(start_date, end_date):
    """Same steps as Section 5's loop, so values match the exported band files."""
    return (
        scene_collection(start_date, end_date)
        .linkCollection(cs_plus, ["cs_cdf"])
        .map(mask_clouds_cs_plus)
        .median()
        .clip(paoay_aoi)
        .select(BANDS)
        .toFloat()                           # avoids integer division in the indices
    )

Calculates indices from the composite. The input bands is the median band composite returned by make_composite(). The output is one three-band image containing NDVI, NDRE, and NDMI.

In [ ]:
def compute_indices(bands):
    """Normalized differences from the median composite; masked where the denominator is 0."""
    layers = []
    for name, (a, b) in INDEX_DEFS.items():
        na, nb = bands.select(a), bands.select(b)
        denom = na.add(nb)
        layers.append(na.subtract(nb).divide(denom).updateMask(denom.neq(0)).rename(name))
    return ee.Image.cat(layers).toFloat()



In [ ]:
def export_to_drive(image, folder, name):
    """Same region and EPSG:32651 grid; masked pixels written as NODATA"""
    task = ee.batch.Export.image.toDrive(
        image=ee.Image(image).toFloat().unmask(NODATA, sameFootprint=False).clip(paoay_aoi),
        description=name,
        folder=folder,
        fileNamePrefix=name,
        region=paoay_aoi.geometry(),
        **GRID,
        maxPixels=1e13,
        fileFormat="GeoTIFF",
        formatOptions={"noData": NODATA},
    )
    task.start()
    return task

## 8.2 Compute indices for all periods

Calculates and export indices for each period.

In [ ]:
try:
    exported_labels = {s["label"] for s in summary if s["status"] == "exported"}
except NameError:
    exported_labels = None
    print("`summary` not found: exporting every period that has imagery.")

index_count = 0
for year, month, start_date, end_date, half in periods_list:
    label = f"{year}_{month:02d}_{half}"

    if exported_labels is not None:
        if label not in exported_labels:
            continue
    elif scene_collection(start_date, end_date).size().getInfo() == 0:
        continue

    indices = compute_indices(make_composite(start_date, end_date))
    export_to_drive(indices, DRIVE_FOLDER_INDICES, f"{INDEX_PREFIX}_{label}")
    index_count += 1

print(f"Started {index_count} index exports → {DRIVE_FOLDER_INDICES}")

## 9. Feature stack for crop classification

Combines the five spectral bands and three indices across all half-months into one multiband raster per year. Feature stack is a GeoTIFF.

- Period slots create the 24 period labels
- Stack names the output bands
- Empty period keeps a placeholder when no scenes are available

In [ ]:
PERIOD_SLOTS = [f"{m:02d}_{h}" for m in range(1, 13) for h in ("H1", "H2")]   # 24
STACK_NAMES = [f"{f}_{slot}" for slot in PERIOD_SLOTS for f in FEATURES]    # 192, e.g. NDVI_03_H1

EMPTY_PERIOD = (
    ee.Image.constant([0] * len(FEATURES))
    .rename(FEATURES)
    .toFloat()
    .updateMask(ee.Image.constant(0))
)


Prepared one period's features and combines one year's periods

In [ ]:
def period_features(start_date, end_date):
    """8 bands for one period: B3 B4 B5 B8 B11 NDVI NDRE NDMI (NoData if no imagery)."""
    has_scenes = scene_collection(start_date, end_date).size().gt(0)
    bands = make_composite(start_date, end_date)
    return ee.Image(ee.Algorithms.If(
        has_scenes,
        bands.addBands(compute_indices(bands)),
        EMPTY_PERIOD,
    ))


def year_feature_stack(year):
    """All 24 periods × 8 features = 192 bands, Jan H1 → Dec H2."""
    periods = [p for p in periods_list if p[0] == year]
    return (
        ee.Image.cat([period_features(start, end) for (_, _, start, end, _) in periods])
        .rename(STACK_NAMES)
        .toFloat()
    )

In [ ]:
print(year_feature_stack(years[0]).bandNames().size().getInfo(), "bands")   # expect 192
print(STACK_NAMES[:10])

Passes the images to drive

In [ ]:
for year in years:
    export_to_drive(year_feature_stack(year), DRIVE_FOLDER_FEATURES, f"{STACK_PREFIX}_{year}")

print(f"Started {len(years)} stack exports → {DRIVE_FOLDER_FEATURES}")

## Check for export task status

In [ ]:
for t in ee.batch.Task.list()[:30]:
    st = t.status()
    print(st["state"], "|", st["description"], "|", st.get("error_message", ""))

In [ ]:
from collections import Counter

tasks = ee.batch.Task.list()
print(Counter(t.status()["state"] for t in tasks[:400]))

for t in tasks[:400]:
    st = t.status()
    if st["state"] == "FAILED":
        print("FAILED |", st["description"], "|", st.get("error_message", ""))